In [71]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [72]:
movies = pd.read_csv("data/tmdb_5000_movies.csv")
credits = pd.read_csv("data/tmdb_5000_credits.csv")

In [73]:
movies = movies.merge(credits, on="title")

In [74]:
movies = movies[
    [
        "movie_id",
        "title",
        "overview",
        "genres",
        "keywords",
        "cast",
        "crew"
    ]
]

In [75]:
movies.dropna(inplace=True)

In [76]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [77]:
movies.shape

(4806, 7)

In [78]:
def convert(text):
    L = []

    for i in ast.literal_eval(text):
        L.append(i['name'])

    return L

In [79]:
movies['genres'] = movies['genres'].apply(convert)

In [80]:
movies['genres'][0]

['Action', 'Adventure', 'Fantasy', 'Science Fiction']

In [81]:
movies['keywords'] = movies['keywords'].apply(convert)

In [82]:
movies['keywords'][0]

['culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d']

In [83]:
def convert_cast(text):

    L = []
    counter = 0

    for i in ast.literal_eval(text):

        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

In [84]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [85]:
movies['cast'][0]

['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver']

In [86]:
def fetch_director(text):

    L = []

    for i in ast.literal_eval(text):

        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L

In [87]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [88]:
movies['crew'][0]

['James Cameron']

In [89]:
movies['genres_display'] = movies['genres']

movies['cast_display'] = movies['cast']

movies['crew_display'] = movies['crew']

In [90]:
movies['crew_display'][0]

['James Cameron']

In [91]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

In [92]:
movies['genres_tags'] = movies['genres'].apply(collapse)

movies['cast_tags'] = movies['cast'].apply(collapse)

movies['crew_tags'] = movies['crew'].apply(collapse)

movies['keywords_tags'] = movies['keywords'].apply(collapse)

In [93]:
print(movies['crew_display'][0])

print(movies['crew_tags'][0])

['James Cameron']
['JamesCameron']


In [94]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [95]:
movies['overview'][0][:10]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to']

In [96]:
movies['tags'] = (
    movies['overview']
    + movies['genres_tags']
    + movies['keywords_tags']
    + movies['cast_tags']
    + movies['crew_tags']
)

In [97]:
movies['tags'][0][:20]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn']

In [98]:
movies['tags'][0][-15:]

['tribe',
 'alienplanet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'loveaffair',
 'antiwar',
 'powerrelations',
 'mindandsoul',
 '3d',
 'SamWorthington',
 'ZoeSaldana',
 'SigourneyWeaver',
 'JamesCameron']

In [99]:
print(movies['crew_display'][0])
print(movies['crew_tags'][0])

['James Cameron']
['JamesCameron']


In [100]:
new_df = movies[
    [
        'movie_id',
        'title',
        'genres_display',
        'cast_display',
        'crew_display',
        'tags'
    ]
]

In [101]:
new_df.head()

,movie_id,title,genres_display,cast_display,crew_display,tags
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],"[John, Carter, is, a, war-weary,, former, mili..."


In [102]:
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [103]:
new_df['tags'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [104]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [105]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [106]:
def stem(text):

    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

In [107]:
new_df['tags'] = new_df['tags'].apply(stem)

In [108]:
cv = CountVectorizer(max_features=5000, stop_words='english')

In [109]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [110]:
vectors.shape

(4806, 5000)

In [111]:
from sklearn.metrics.pairwise import cosine_similarity

In [120]:
similarity = cosine_similarity(vectors).astype("float16")

In [113]:
similarity.shape

(4806, 4806)

In [114]:
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)

In [115]:
recommend("Avatar")

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [116]:
import os
os.makedirs("model", exist_ok=True)

In [117]:
import pickle

pickle.dump(new_df, open("model/movie_list.pkl", "wb"))
pickle.dump(similarity, open("model/similarity.pkl", "wb"))

In [118]:
import pickle

movies = pickle.load(open("model/movie_list.pkl", "rb"))

print(movies.iloc[0]["crew_display"])

['James Cameron']


In [119]:
print(new_df.columns)

Index(['movie_id', 'title', 'genres_display', 'cast_display', 'crew_display',
       'tags'],
      dtype='str')
